# Work with VneuroTK paths

VneuroTK path objects describe locations without performing I/O. They accept `str` or `pathlib.Path` roots and produce `pathlib.Path` values through `.fpath`.

| Class | Use |
|---|---|
| `VTKPath` | VneuroTK HDF5 recordings and the base naming convention |
| `EphysPath` | Session-level electrophysiology data |
| `MNEPath` | MNE-readable MEG/EEG files with a flat filename under `root` |
| `BIDSPath` | BIDS entities backed by the optional `mne-bids` package |

This notebook uses path construction only and does not read files. Full BIDS loading requires `vneurotk[mne]`.

<a href="../path.ipynb" download>Download this notebook</a>


## Construct electrophysiology paths

In [1]:
from pathlib import Path

from vneurotk.io import BIDSPath, EphysPath, MNEPath, VTKPath

root = Path("data")
ephys = EphysPath(
    root=root,
    session_id="251024_FanFan_nsd1w_MSB",
    dtype="TrialRaster",
    extension="h5",
)
ephys.fpath

PosixPath('data/sessions/251024_FanFan_nsd1w_MSB/TrialRaster_251024_FanFan_nsd1w_MSB.h5')

`EphysPath` supports unit- and channel-level products such as `TrialRaster`, `MeanFr`, `ChTrialRaster`, and `ChStimFr`. A `probe` adds the probe suffix; `from_components()` assembles the session identifier.

In [2]:
multi_probe = EphysPath(
    root=root,
    session_id="251024_FanFan_nsd1w_MSB",
    dtype="TrialRaster",
    probe=0,
    extension="h5",
)
from_parts = EphysPath.from_components(
    root=root,
    date="251024",
    subject="FanFan",
    paradigm="nsd1w",
    region="MSB",
    dtype="TrialRaster",
    extension="h5",
)
from_parts.session_dir, multi_probe.fpath

(PosixPath('data/sessions/251024_FanFan_nsd1w_MSB'),
 PosixPath('data/sessions/251024_FanFan_nsd1w_MSB/TrialRaster_251024_FanFan_nsd1w_MSB_probe0.h5'))

## Construct MNE and BIDS paths

In [3]:
mne_path = MNEPath(
    root=root,
    subject="01",
    session="ImageNet01",
    task="ImageNet",
    run="01",
    suffix="meg_clean",
    extension=".fif",
)

bids_path = BIDSPath(
    root=root,
    subject="01",
    session="01",
    task="images",
    run="01",
    suffix="meg",
    extension=".fif",
)
mne_path.fpath, bids_path.fpath

2026-08-01 17:59:14.145 | WARNING  | vneurotk.io.path:__post_init__:425 - mne_bids not available, BIDSPath functionality limited


(PosixPath('data/sub-01_ses-ImageNet01_task-ImageNet_run-01_meg_clean.fif'),
 PosixPath('data/sub-01_ses-01_task-images_run-01_meg.fif'))

`MNEPath` builds a filename directly under `root`. `BIDSPath` delegates BIDS layout and entities to `mne_bids.BIDSPath` when the extra is installed; otherwise `.bids_path` is `None` and `.fpath` uses the base fallback.

## Construct a VneuroTK output path

In [4]:
output = VTKPath(
    Path("outputs"),
    subject="01",
    session="ImageNet01",
    task="ImageNet",
    run="01",
)
output.fpath

PosixPath('outputs/sub-01_ses-ImageNet01_task-ImageNet_run-01.h5')

Pass a directory as the `VTKPath` root; the object appends the recording filename. To read an existing HDF5 file, pass that file path directly to `vneurotk.read()` instead of wrapping it as a `VTKPath` root.

## Related documentation

- [Build neural data](data.ipynb)
- [I/O API](../api/io.md)
- [VneuroTK HDF5 format](../format/hdf5.md)
